In [33]:
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *

In [34]:
import random
import os

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [35]:
df = pd.read_csv('Meshva_dataset.csv')
print(f"Columns: {df.columns.tolist()}")
TARGET = 'generated_power_kw'
DROP_COLS = [
    'wind_speed_80_m_above_gnd',
    'wind_direction_80_m_above_gnd',
    'wind_speed_900_mb',
    'wind_direction_900_mb',
    'wind_gust_10_m_above_gnd'
]
df = df.drop(columns=DROP_COLS)

y = df[TARGET]
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

features_num = X_train.shape[1]
print(f"Number of features: {features_num}")


Columns: ['temperature_2_m_above_gnd', 'relative_humidity_2_m_above_gnd', 'mean_sea_level_pressure_MSL', 'total_precipitation_sfc', 'snowfall_amount_sfc', 'total_cloud_cover_sfc', 'high_cloud_cover_high_cld_lay', 'medium_cloud_cover_mid_cld_lay', 'low_cloud_cover_low_cld_lay', 'shortwave_radiation_backwards_sfc', 'wind_speed_10_m_above_gnd', 'wind_direction_10_m_above_gnd', 'wind_speed_80_m_above_gnd', 'wind_direction_80_m_above_gnd', 'wind_speed_900_mb', 'wind_direction_900_mb', 'wind_gust_10_m_above_gnd', 'angle_of_incidence', 'zenith', 'azimuth', 'generated_power_kw']
Number of features: 15


In [36]:
model = keras.Sequential([
    layers.Input(shape=(features_num,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)

history = model.fit(X_train, y_train, validation_split=0.2, epochs=200, batch_size=32)

Epoch 1/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 2193284.5000 - rmse: 1480.9741 - val_loss: 2016088.2500 - val_rmse: 1419.8903
Epoch 2/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 2177768.0000 - rmse: 1475.7263 - val_loss: 1992481.7500 - val_rmse: 1411.5530
Epoch 3/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 2133144.2500 - rmse: 1460.5288 - val_loss: 1933930.5000 - val_rmse: 1390.6583
Epoch 4/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 2044596.6250 - rmse: 1429.8939 - val_loss: 1835043.0000 - val_rmse: 1354.6376
Epoch 5/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1916796.0000 - rmse: 1384.4840 - val_loss: 1710422.7500 - val_rmse: 1307.8313
Epoch 6/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 1774467.2500 - rmse: 1332.0913 - val_loss: 1589638.6250 - val_rmse: 1260.8087
Epoch 7/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1652698.7500 - rmse: 1285.5734 - val_loss: 1501186.7500 - val_rmse: 1225.2292
Epoch 8/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 1

In [37]:
y_true = y_test                          # already kW
y_pred = model.predict(X_test).flatten() # already kW

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)
mbe  = np.mean(y_pred - y_true)

print(f"RMSE: {rmse:.2f} kW")
print(f"MAE:  {mae:.2f} kW")
print(f"R²:   {r2:.4f}")
print(f"MBE:  {mbe:.2f} kW")

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
RMSE: 545.88 kW
MAE:  410.30 kW
R²:   0.6738
MBE:  0.28 kW
